# Problem 5 - Phan tich kinh doanh theo dia ly
Phan tich Region/City/District, thi phan, tang truong, don vi van hanh va ma tran chien luoc tren du lieu Silver.

In [ ]:
from pathlib import Path
import warnings
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def locate_repo():
    for start in [Path.cwd().resolve(), Path(r'C:/dpbngoc/DA & AI/DAAI_N2.3')]:
        for path in [start, *start.parents]:
            if (path / 'silver_data').exists(): return path
    raise FileNotFoundError('Khong tim thay silver_data')

REPO_ROOT = locate_repo()
def find_data_file(filename):
    paths = sorted((REPO_ROOT / 'silver_data').glob(f'.silver_pipeline_work_*/excel/{filename}'), reverse=True)
    for path in paths + [REPO_ROOT / filename]:
        if path.exists(): return path
    raise FileNotFoundError(filename)

def show_result(df, name, index=False):
    print(f'\n{"="*100}\n{name.replace("_"," " ).replace(".csv","").upper()}\n{"="*100}')
    display(df)

def summarize(data, levels):
    result = data.groupby(levels, as_index=False).agg(Revenue=('NetRevenue','sum'), Orders=('order_id','nunique'), Customers=('customer_id','nunique'), Quantity=('quantity','sum'))
    result['AOV'] = result['Revenue'] / result['Orders'].replace(0,np.nan)
    result['UnitsPerOrder'] = result['Quantity'] / result['Orders'].replace(0,np.nan)
    result['RevenuePerCustomer'] = result['Revenue'] / result['Customers'].replace(0,np.nan)
    result['GlobalRevenueShare_%'] = result['Revenue'] / data['NetRevenue'].sum() * 100
    if len(levels) > 1:
        parents = levels[:-1]
        result['ShareWithinParent_%'] = result['Revenue'] / result.groupby(parents)['Revenue'].transform('sum') * 100
    else:
        result['ShareWithinParent_%'] = result['GlobalRevenueShare_%']
    return result

def add_growth(data, result, levels, current_periods, previous_periods):
    periods = data['order_date'].dt.to_period('M')
    current = data[periods.isin(current_periods)].groupby(levels)['NetRevenue'].sum().rename('Current4MRevenue')
    previous = data[periods.isin(previous_periods)].groupby(levels)['NetRevenue'].sum().rename('Previous4MRevenue')
    result = result.merge(current, on=levels, how='left').merge(previous, on=levels, how='left')
    result[['Current4MRevenue','Previous4MRevenue']] = result[['Current4MRevenue','Previous4MRevenue']].fillna(0)
    result['Growth_%'] = (result['Current4MRevenue'] / result['Previous4MRevenue'].replace(0,np.nan) - 1) * 100
    return result

def strategic_matrix(table, level_name):
    result = table.copy()
    share_cut = result['GlobalRevenueShare_%'].median()
    growth_cut = result['Growth_%'].median(skipna=True)
    high_share = result['GlobalRevenueShare_%'] >= share_cut
    high_growth = result['Growth_%'].fillna(-np.inf) >= growth_cut
    result['StrategicQuadrant'] = np.select([high_share & high_growth, high_share & ~high_growth, ~high_share & high_growth], ['Dan dau - tang truong','Quy mo lon - tang truong cham','Moi noi - tang truong cao'], default='Thi phan thap - tang truong cham')
    result['Recommendation'] = result['StrategicQuadrant'].map({
        'Dan dau - tang truong':'Uu tien dau tu ton kho, marketing va mo rong phu song',
        'Quy mo lon - tang truong cham':'Bao ve thi phan, tang retention va toi uu AOV',
        'Moi noi - tang truong cao':'Thu nghiem mo rong co kiem soat; tang nhan dien dia phuong',
        'Thi phan thap - tang truong cham':'Ra soat san pham, kenh ban va chi phi; chi dau tu khi co unit economics tot'})
    result['Level'] = level_name
    return result

def main():
    orders = pd.read_csv(find_data_file('orders_enriched.csv'), low_memory=False)
    items = pd.read_csv(find_data_file('order_items.csv'), low_memory=False)
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
    orders['order_status'] = orders['order_status'].astype(str).str.lower().str.strip()
    orders = orders[orders['order_status'].eq('delivered')].copy()
    for col in ['region','city','district']:
        orders[col] = orders[col].fillna('Unknown').astype(str).str.strip()
    if 'office_id' in orders.columns:
        orders['OfficeID'] = orders['office_id'].astype(str)
        orders['OfficeSource'] = 'source_office_id'
    else:
        orders['OfficeID'] = 'GEO|' + orders['region'] + '|' + orders['city'] + '|' + orders['district']
        orders['OfficeSource'] = 'geographic_proxy_city_district'
    items[['quantity','unit_price','discount_amount']] = items[['quantity','unit_price','discount_amount']].apply(pd.to_numeric, errors='coerce')
    items['NetRevenue'] = (items['quantity'] * items['unit_price'] - items['discount_amount'].fillna(0)).clip(lower=0)
    data = items.merge(orders[['order_id','order_date','customer_id','region','city','district','OfficeID','OfficeSource']], on='order_id', how='inner', validate='many_to_one')
    hierarchy = data[['region','city','district']].drop_duplicates().sort_values(['region','city','district'])
    show_result(hierarchy, 'geography_hierarchy')

    end_period = data['order_date'].max().to_period('M')
    current_periods = pd.period_range(end=end_period, periods=4, freq='M')
    previous_periods = pd.period_range(end=end_period-4, periods=4, freq='M')
    region = add_growth(data, summarize(data,['region']), ['region'], current_periods, previous_periods)
    city = add_growth(data, summarize(data,['region','city']), ['region','city'], current_periods, previous_periods)
    district = add_growth(data, summarize(data,['region','city','district']), ['region','city','district'], current_periods, previous_periods)
    show_result(region, 'region_strategy_metrics'); show_result(city, 'city_strategy_metrics'); show_result(district, 'district_strategy_metrics')

    quantity_compare = pd.concat([
        region.assign(Level='Region', Geography=region['region']),
        city.assign(Level='City', Geography=city['city']),
        district.assign(Level='District', Geography=district['district'])], ignore_index=True, sort=False)
    quantity_compare = quantity_compare[['Level','Geography','region','city','district','Quantity','Orders','UnitsPerOrder','Revenue']].sort_values(['Level','Quantity'], ascending=[True,False])
    show_result(quantity_compare, 'quantity_comparison_all_levels')

    order_geo = data.groupby(['region','city','OfficeID','OfficeSource','customer_id','order_id'], as_index=False).agg(Revenue=('NetRevenue','sum'), Quantity=('quantity','sum'), OrderDate=('order_date','first'))
    office = order_geo.groupby(['region','city','OfficeID','OfficeSource'], as_index=False).agg(Revenue=('Revenue','sum'), Invoices=('order_id','nunique'), Customers=('customer_id','nunique'), Quantity=('Quantity','sum'))
    office['AOV'] = office['Revenue'] / office['Invoices'].replace(0,np.nan)
    office['RevenuePerCustomer'] = office['Revenue'] / office['Customers'].replace(0,np.nan)
    repeat = order_geo.groupby(['region','city','OfficeID','OfficeSource','customer_id'])['order_id'].nunique().ge(2).groupby(level=[0,1,2,3]).mean().mul(100).rename('RepeatRate_%').reset_index()
    office = office.merge(repeat, on=['region','city','OfficeID','OfficeSource'], how='left')
    office['RevenueShareInCity_%'] = office['Revenue'] / office.groupby(['region','city'])['Revenue'].transform('sum') * 100
    office_period = order_geo.copy(); office_period['Period'] = office_period['OrderDate'].dt.to_period('M')
    current_office = office_period[office_period['Period'].isin(current_periods)].groupby(['region','city','OfficeID'])['Revenue'].sum().rename('Current4MRevenue')
    previous_office = office_period[office_period['Period'].isin(previous_periods)].groupby(['region','city','OfficeID'])['Revenue'].sum().rename('Previous4MRevenue')
    office = office.merge(current_office, on=['region','city','OfficeID'], how='left').merge(previous_office, on=['region','city','OfficeID'], how='left')
    office[['Current4MRevenue','Previous4MRevenue']] = office[['Current4MRevenue','Previous4MRevenue']].fillna(0)
    office['Growth_%'] = (office['Current4MRevenue'] / office['Previous4MRevenue'].replace(0,np.nan) - 1) * 100
    office['RankInCity'] = office.groupby(['region','city'])['Revenue'].rank(method='dense', ascending=False).astype(int)
    office['Recommendation'] = np.select([
        (office['RankInCity'] == 1) & (office['Growth_%'] >= 0),
        office['Growth_%'] < 0, office['RepeatRate_%'] < office['RepeatRate_%'].median()],
        ['Duy tri dau tu; chia se best practice trong cung thanh pho',
         'Kiem tra nhu cau, kenh ban va danh muc; lap ke hoach phuc hoi',
         'Tang retention, CRM va chuong trinh mua lai'],
        default='Duy tri van hanh va toi uu AOV')
    show_result(office.sort_values(['region','city','RankInCity']), 'office_agency_comparison')

    region_matrix = strategic_matrix(region, 'Region')
    city_matrix = strategic_matrix(city, 'City')
    show_result(region_matrix, 'region_growth_share_matrix'); show_result(city_matrix, 'city_growth_share_matrix')
    strategic = pd.concat([region_matrix[['Level','region','GlobalRevenueShare_%','Growth_%','StrategicQuadrant','Recommendation']], city_matrix[['Level','region','city','GlobalRevenueShare_%','Growth_%','StrategicQuadrant','Recommendation']]], ignore_index=True, sort=False)
    show_result(strategic, 'geography_strategic_recommendations')

    plt.figure(figsize=(8,6)); plt.scatter(region_matrix['GlobalRevenueShare_%'], region_matrix['Growth_%'], s=region_matrix['Revenue']/region_matrix['Revenue'].max()*700+80)
    for _, row in region_matrix.iterrows(): plt.annotate(row['region'], (row['GlobalRevenueShare_%'], row['Growth_%']))
    plt.axvline(region_matrix['GlobalRevenueShare_%'].median(), color='gray', linestyle='--'); plt.axhline(region_matrix['Growth_%'].median(), color='gray', linestyle='--')
    plt.xlabel('Thi phan doanh thu (%)'); plt.ylabel('Tang truong 4M vs 4M (%)'); plt.title('Ma tran thi phan - tang truong theo Region')
    plt.tight_layout(); plt.show()
    region.sort_values('Revenue').plot(kind='barh', x='region', y=['Revenue','Quantity'], subplots=True, figsize=(10,7), legend=False, title=['Doanh thu theo Region','So luong theo Region'])
    plt.tight_layout(); plt.show()
    city.head(15).sort_values('Revenue').plot(kind='barh', x='city', y='Revenue', figsize=(10,6), legend=False, title='Top City theo doanh thu')
    plt.tight_layout(); plt.show()
    plt.figure(figsize=(9,6)); plt.scatter(office['Invoices'], office['Revenue'], c=office['RepeatRate_%'], cmap='plasma', alpha=.65)
    plt.xlabel('So hoa don'); plt.ylabel('Doanh thu'); plt.title('Hieu qua Office/Agency trong cung dia ban'); plt.colorbar(label='Repeat Rate (%)')
    plt.tight_layout(); plt.show()
    print('Hoan tat Problem 5 | Region:', len(region), '| City:', len(city), '| Office/proxy:', len(office))
    print('Office source:', ', '.join(office['OfficeSource'].unique())); print('Tat ca bang va bieu do da hien thi truc tiep.')
    return region, city, district, office, strategic

region_result, city_result, district_result, office_result, geography_strategy = main()
